# TomTom Global Pooled Empirical Quantiles — Final RKP Notebook

**Konteks:** DT-MCVRPTW–TD Risk-Aware  
**Tujuan:** mengestimasi global traffic multiplier p50, p85, dan p95 dari rasio TomTom historical traffic travel time terhadap no-traffic travel time.

Rasio perjalanan didefinisikan sebagai:

\[
R_{ij,t}=
\frac{TT_{ij,t}^{historic}}
{TT_{ij,t}^{noTraffic}}
\]

Seluruh rasio dari pasangan origin–destination dan waktu keberangkatan representatif digabungkan menjadi satu distribusi empiris global:

\[
\mathcal{R}=\{R_{ij,t}\}
\]

Normalized pooled empirical quantile multiplier dihitung sebagai:

\[
M_q=
\frac{Q_q(\mathcal{R})}
{Q_{0.50}(\mathcal{R})}
\]

Dengan demikian, \(M_{50}=1.0000\).

Notebook ini mendukung dua input:

1. **ZIP/CSV hasil ekstraksi TomTom sebelumnya** untuk mereproduksi hasil RKP secara tepat.
2. **Excel baseline dataset** untuk menjalankan ekstraksi TomTom API kembali menggunakan desain sampling yang sama.

> Untuk reproduksi angka pada draft RKP, gunakan `TomTom_External_Validation_Output.zip` yang menyimpan 960 observasi raw TomTom.

## 1. Instalasi dan import library

In [ ]:
!pip -q install openpyxl requests tqdm

import os
import io
import json
import time
import hashlib
import zipfile
import shutil
import getpass
import warnings

from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Library siap.")

## 2. Konfigurasi desain empiris

In [ ]:
# ============================================================
# SAMPLING DESIGN
# ============================================================
RANDOM_SEED = 42
N_DISTANCE_STRATA = 5
CUSTOMERS_PER_STRATUM = 2

# Jadwal yang digunakan pada output RKP.
REPRESENTATIVE_DATES = [
    "2026-07-15",  # Wednesday
    "2026-07-17",  # Friday
    "2026-07-18",  # Saturday
    "2026-07-20",  # Monday
]

DEPARTURE_HOURS = [7, 8, 9, 10, 12, 14, 16, 18]

TRAVEL_MODE = "car"
ROUTE_TYPE = "fastest"
REQUEST_PAUSE_SEC = 0.05

BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_SEED = 2026

OUTPUT_DIR = Path("tomtom_global_pooled_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Expected OD pairs       :", 30)
print("Departure time points   :", len(REPRESENTATIVE_DATES) * len(DEPARTURE_HOURS))
print("Expected observations   :", 30 * len(REPRESENTATIVE_DATES) * len(DEPARTURE_HOURS))
print("Bootstrap iterations    :", BOOTSTRAP_ITERATIONS)
print("Output directory        :", OUTPUT_DIR.resolve())

## 3. Upload input

Upload **salah satu** file berikut:

- `TomTom_External_Validation_Output.zip`, atau
- `tomtom_raw_validation.csv`, atau
- file Excel baseline yang memiliki sheet `Stops` dan `Depot`.

Mode ZIP/CSV digunakan untuk mereproduksi hasil empiris RKP.  
Mode Excel akan menjalankan ekstraksi TomTom API kembali.

In [ ]:
try:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Tidak ada file yang di-upload.")

    INPUT_FILE = Path(next(iter(uploaded.keys())))

except ImportError:
    INPUT_FILE = Path(
        input("Masukkan path ZIP/CSV/XLSX: ").strip()
    )

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {INPUT_FILE}")

print("Input:", INPUT_FILE)

## 4. Deteksi mode input dan baca raw TomTom bila tersedia

In [ ]:
def sha256_file(file_path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


SOURCE_SHA256 = sha256_file(INPUT_FILE)


def locate_raw_tomtom(input_path: Path):
    suffix = input_path.suffix.lower()

    if suffix == ".csv":
        return input_path

    if suffix == ".zip":
        extract_dir = OUTPUT_DIR / "input_extracted"
        extract_dir.mkdir(exist_ok=True)

        with zipfile.ZipFile(input_path, "r") as zf:
            zf.extractall(extract_dir)

        candidates = list(
            extract_dir.rglob("tomtom_raw_validation.csv")
        )

        if not candidates:
            raise FileNotFoundError(
                "tomtom_raw_validation.csv tidak ditemukan di ZIP."
            )

        return candidates[0]

    return None


RAW_INPUT = locate_raw_tomtom(INPUT_FILE)

if RAW_INPUT is not None:
    INPUT_MODE = "REPRODUCE_FROM_RAW"
    raw = pd.read_csv(RAW_INPUT)

    print("Mode:", INPUT_MODE)
    print("Raw TomTom:", RAW_INPUT)
    print("Rows:", len(raw))

else:
    INPUT_MODE = "EXTRACT_FROM_API"
    raw = None

    print("Mode:", INPUT_MODE)
    print("Notebook akan membaca sheet Stops dan Depot.")

## 5. Ekstraksi TomTom API bila input berupa Excel

In [ ]:
if INPUT_MODE == "EXTRACT_FROM_API":
    xls = pd.ExcelFile(INPUT_FILE)

    required_sheets = {"Stops", "Depot"}
    missing_sheets = required_sheets - set(xls.sheet_names)

    if missing_sheets:
        raise ValueError(
            f"Sheet wajib tidak ditemukan: {sorted(missing_sheets)}"
        )

    stops = pd.read_excel(INPUT_FILE, sheet_name="Stops")
    depot = pd.read_excel(INPUT_FILE, sheet_name="Depot")

    required_stops = {
        "customer_code",
        "customer_name",
        "longitude",
        "latitude",
    }

    required_depot = {
        "depot_id",
        "depot_name",
        "longitude",
        "latitude",
    }

    if required_stops - set(stops.columns):
        raise ValueError(
            "Kolom sheet Stops tidak sesuai kebutuhan notebook."
        )

    if required_depot - set(depot.columns):
        raise ValueError(
            "Kolom sheet Depot tidak sesuai kebutuhan notebook."
        )

    print("Stops rows:", len(stops))
    print("Depot rows:", len(depot))

else:
    print("Lewati pembacaan Excel karena raw TomTom telah tersedia.")

In [ ]:
if INPUT_MODE == "EXTRACT_FROM_API":

    def haversine_km(lat1, lon1, lat2, lon2):
        r = 6371.0088

        lat1 = np.radians(lat1)
        lon1 = np.radians(lon1)
        lat2 = np.radians(lat2)
        lon2 = np.radians(lon2)

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = (
            np.sin(dlat / 2.0) ** 2
            + np.cos(lat1)
            * np.cos(lat2)
            * np.sin(dlon / 2.0) ** 2
        )

        return 2 * r * np.arcsin(np.sqrt(a))


    for df_name, df in [("Stops", stops), ("Depot", depot)]:
        df["longitude"] = pd.to_numeric(
            df["longitude"], errors="coerce"
        )
        df["latitude"] = pd.to_numeric(
            df["latitude"], errors="coerce"
        )

        invalid = (
            df["longitude"].isna()
            | df["latitude"].isna()
            | ~df["longitude"].between(-180, 180)
            | ~df["latitude"].between(-90, 90)
        )

        if invalid.any():
            raise ValueError(
                f"{df_name}: {int(invalid.sum())} koordinat tidak valid."
            )


    depot_row = depot.iloc[0].copy()

    customers = (
        stops[
            [
                "customer_code",
                "customer_name",
                "longitude",
                "latitude",
            ]
        ]
        .dropna(
            subset=[
                "customer_code",
                "longitude",
                "latitude",
            ]
        )
        .drop_duplicates(subset=["customer_code"])
        .copy()
    )

    customers["radial_distance_km"] = haversine_km(
        float(depot_row["latitude"]),
        float(depot_row["longitude"]),
        customers["latitude"].to_numpy(),
        customers["longitude"].to_numpy(),
    )

    customers["distance_stratum"] = pd.qcut(
        customers["radial_distance_km"].rank(method="first"),
        q=N_DISTANCE_STRATA,
        labels=[
            f"S{i + 1}"
            for i in range(N_DISTANCE_STRATA)
        ],
    )

    sampled_parts = []

    for stratum, group in customers.groupby(
        "distance_stratum",
        observed=True,
    ):
        n_take = min(
            CUSTOMERS_PER_STRATUM,
            len(group),
        )

        sampled_parts.append(
            group.sample(
                n=n_take,
                random_state=RANDOM_SEED,
            )
        )

    sampled_customers = (
        pd.concat(sampled_parts, ignore_index=True)
        .sort_values(
            [
                "distance_stratum",
                "radial_distance_km",
            ]
        )
        .reset_index(drop=True)
    )

    display(
        sampled_customers[
            [
                "customer_code",
                "customer_name",
                "radial_distance_km",
                "distance_stratum",
            ]
        ]
    )

    print("Sampled customers:", len(sampled_customers))

else:
    print("Lewati sampling pelanggan karena raw TomTom telah tersedia.")

In [ ]:
if INPUT_MODE == "EXTRACT_FROM_API":

    def make_pair(
        pair_id,
        pair_type,
        from_id,
        from_name,
        from_lon,
        from_lat,
        to_id,
        to_name,
        to_lon,
        to_lat,
    ):
        return {
            "pair_id": pair_id,
            "pair_type": pair_type,
            "from_id": str(from_id),
            "from_name": str(from_name),
            "from_lon": float(from_lon),
            "from_lat": float(from_lat),
            "to_id": str(to_id),
            "to_name": str(to_name),
            "to_lon": float(to_lon),
            "to_lat": float(to_lat),
            "haversine_km": float(
                haversine_km(
                    float(from_lat),
                    float(from_lon),
                    float(to_lat),
                    float(to_lon),
                )
            ),
        }


    pairs = []

    # Depot -> customer
    for i, row in sampled_customers.iterrows():
        pairs.append(
            make_pair(
                f"D2C_{i + 1:02d}",
                "depot_to_customer",
                depot_row["depot_id"],
                depot_row["depot_name"],
                depot_row["longitude"],
                depot_row["latitude"],
                row["customer_code"],
                row["customer_name"],
                row["longitude"],
                row["latitude"],
            )
        )

    # Customer -> depot
    for i, row in sampled_customers.iterrows():
        pairs.append(
            make_pair(
                f"C2D_{i + 1:02d}",
                "customer_to_depot",
                row["customer_code"],
                row["customer_name"],
                row["longitude"],
                row["latitude"],
                depot_row["depot_id"],
                depot_row["depot_name"],
                depot_row["longitude"],
                depot_row["latitude"],
            )
        )

    # Customer -> customer
    customer_order = sampled_customers.sample(
        frac=1.0,
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)

    for i in range(len(customer_order)):
        from_row = customer_order.iloc[i]
        to_row = customer_order.iloc[
            (i + 1) % len(customer_order)
        ]

        pairs.append(
            make_pair(
                f"C2C_{i + 1:02d}",
                "customer_to_customer",
                from_row["customer_code"],
                from_row["customer_name"],
                from_row["longitude"],
                from_row["latitude"],
                to_row["customer_code"],
                to_row["customer_name"],
                to_row["longitude"],
                to_row["latitude"],
            )
        )

    od_pairs = pd.DataFrame(pairs)

    if (od_pairs["from_id"] == od_pairs["to_id"]).any():
        raise ValueError("Terdapat self-loop pada OD sample.")

    print("Total OD pairs:", len(od_pairs))
    display(od_pairs)

else:
    od_pairs = (
        raw[
            [
                "pair_id",
                "pair_type",
                "from_id",
                "from_name",
                "from_lon",
                "from_lat",
                "to_id",
                "to_name",
                "to_lon",
                "to_lat",
                "haversine_km",
            ]
        ]
        .drop_duplicates(subset=["pair_id"])
        .reset_index(drop=True)
    )

    print("OD pairs from raw:", len(od_pairs))

In [ ]:
if INPUT_MODE == "EXTRACT_FROM_API":

    departure_schedule = []

    for date_str in REPRESENTATIVE_DATES:
        d = pd.Timestamp(date_str)

        for hour in DEPARTURE_HOURS:
            departure_schedule.append(
                datetime(
                    d.year,
                    d.month,
                    d.day,
                    hour,
                    0,
                    0,
                    tzinfo=timezone(
                        timedelta(hours=7)
                    ),
                )
            )

    schedule_df = pd.DataFrame(
        {
            "depart_at": [
                dt.isoformat()
                for dt in departure_schedule
            ],
            "weekday": [
                dt.strftime("%A")
                for dt in departure_schedule
            ],
            "hour": [
                dt.hour
                for dt in departure_schedule
            ],
        }
    )

    display(schedule_df)

    print(
        "Estimated API calls:",
        len(od_pairs) * len(departure_schedule),
    )

else:
    departure_schedule = sorted(
        pd.to_datetime(raw["depart_at"]).tolist()
    )

    print(
        "Unique departure timestamps:",
        raw["depart_at"].nunique(),
    )

In [ ]:
if INPUT_MODE == "EXTRACT_FROM_API":

    TOMTOM_API_KEY = getpass.getpass(
        "Masukkan TomTom API Key: "
    ).strip()

    if not TOMTOM_API_KEY:
        raise ValueError("API key kosong.")

    print("API key diterima dan tidak ditampilkan.")

else:
    TOMTOM_API_KEY = None
    print("API key tidak diperlukan pada mode reproduksi raw.")

In [ ]:
if INPUT_MODE == "EXTRACT_FROM_API":

    BASE_URL = (
        "https://api.tomtom.com/routing/1/calculateRoute"
    )

    retry_strategy = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    session = requests.Session()
    adapter = HTTPAdapter(
        max_retries=retry_strategy
    )
    session.mount("https://", adapter)


    def call_tomtom_route(
        pair,
        depart_at,
        timeout=30,
    ):
        route_locations = (
            f"{pair['from_lat']},{pair['from_lon']}:"
            f"{pair['to_lat']},{pair['to_lon']}"
        )

        url = (
            f"{BASE_URL}/"
            f"{route_locations}/json"
        )

        params = {
            "key": TOMTOM_API_KEY,
            "traffic": "true",
            "routeType": ROUTE_TYPE,
            "travelMode": TRAVEL_MODE,
            "computeTravelTimeFor": "all",
            "routeRepresentation": "summaryOnly",
            "departAt": depart_at.isoformat(),
        }

        response = session.get(
            url,
            params=params,
            timeout=timeout,
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"TomTom HTTP "
                f"{response.status_code}: "
                f"{response.text[:500]}"
            )

        payload = response.json()

        if not payload.get("routes"):
            raise RuntimeError(
                "Response TomTom tidak memiliki routes."
            )

        summary = payload["routes"][0]["summary"]

        required_fields = [
            "travelTimeInSeconds",
            "noTrafficTravelTimeInSeconds",
            "historicTrafficTravelTimeInSeconds",
        ]

        missing = [
            field
            for field in required_fields
            if field not in summary
        ]

        if missing:
            raise RuntimeError(
                f"Field TomTom tidak lengkap: {missing}"
            )

        return summary


    test_pair = od_pairs.iloc[0].to_dict()
    test_departure = departure_schedule[0]

    test_summary = call_tomtom_route(
        test_pair,
        test_departure,
    )

    print("TEST REQUEST BERHASIL")
    print(
        json.dumps(
            {
                key: value
                for key, value in test_summary.items()
                if key
                in [
                    "lengthInMeters",
                    "travelTimeInSeconds",
                    "trafficDelayInSeconds",
                    "noTrafficTravelTimeInSeconds",
                    "historicTrafficTravelTimeInSeconds",
                ]
            },
            indent=2,
        )
    )

else:
    print("Lewati test request API.")

In [ ]:
if INPUT_MODE == "EXTRACT_FROM_API":

    RAW_CACHE_FILE = (
        OUTPUT_DIR / "tomtom_raw_validation.csv"
    )

    if RAW_CACHE_FILE.exists():
        raw_cache = pd.read_csv(RAW_CACHE_FILE)

        completed_keys = set(
            zip(
                raw_cache["pair_id"].astype(str),
                raw_cache["depart_at"].astype(str),
            )
        )

        records = raw_cache.to_dict("records")

        print(
            "Cache ditemukan:",
            len(raw_cache),
            "rows",
        )

    else:
        completed_keys = set()
        records = []

        print("Belum ada cache.")


    errors = []

    tasks = [
        (pair, depart_at)
        for _, pair in od_pairs.iterrows()
        for depart_at in departure_schedule
    ]


    for pair, depart_at in tqdm(
        tasks,
        desc="TomTom requests",
    ):
        key = (
            str(pair["pair_id"]),
            depart_at.isoformat(),
        )

        if key in completed_keys:
            continue

        try:
            summary = call_tomtom_route(
                pair.to_dict(),
                depart_at,
            )

            no_traffic = float(
                summary[
                    "noTrafficTravelTimeInSeconds"
                ]
            )

            historic = float(
                summary[
                    "historicTrafficTravelTimeInSeconds"
                ]
            )

            best_estimate = float(
                summary["travelTimeInSeconds"]
            )

            if no_traffic <= 0:
                raise ValueError(
                    "noTrafficTravelTimeInSeconds <= 0"
                )

            record = {
                "pair_id": str(pair["pair_id"]),
                "pair_type": str(pair["pair_type"]),
                "from_id": str(pair["from_id"]),
                "from_name": str(pair["from_name"]),
                "from_lon": float(pair["from_lon"]),
                "from_lat": float(pair["from_lat"]),
                "to_id": str(pair["to_id"]),
                "to_name": str(pair["to_name"]),
                "to_lon": float(pair["to_lon"]),
                "to_lat": float(pair["to_lat"]),
                "haversine_km": float(
                    pair["haversine_km"]
                ),
                "depart_at": depart_at.isoformat(),
                "weekday": depart_at.strftime("%A"),
                "hour": depart_at.hour,
                "length_m": float(
                    summary.get(
                        "lengthInMeters",
                        np.nan,
                    )
                ),
                "travel_time_sec": best_estimate,
                "traffic_delay_sec": float(
                    summary.get(
                        "trafficDelayInSeconds",
                        np.nan,
                    )
                ),
                "no_traffic_sec": no_traffic,
                "historic_traffic_sec": historic,
                "historic_ratio": (
                    historic / no_traffic
                ),
                "best_estimate_ratio": (
                    best_estimate / no_traffic
                ),
                "api_version": "Routing API v1",
                "travel_mode": TRAVEL_MODE,
                "route_type": ROUTE_TYPE,
            }

            records.append(record)
            completed_keys.add(key)

            pd.DataFrame(records).to_csv(
                RAW_CACHE_FILE,
                index=False,
            )

        except Exception as exc:
            errors.append(
                {
                    "pair_id": str(pair["pair_id"]),
                    "depart_at": (
                        depart_at.isoformat()
                    ),
                    "error": str(exc),
                }
            )

        time.sleep(REQUEST_PAUSE_SEC)


    raw = pd.DataFrame(records)

    if errors:
        pd.DataFrame(errors).to_csv(
            OUTPUT_DIR
            / "tomtom_request_errors.csv",
            index=False,
        )

    print("Successful rows:", len(raw))
    print("Errors:", len(errors))

else:
    print("Raw TomTom sudah dimuat dari input.")

## 6. Audit coverage dan kualitas observasi

In [ ]:
required_columns = {
    "pair_id",
    "pair_type",
    "depart_at",
    "hour",
    "no_traffic_sec",
    "historic_traffic_sec",
    "historic_ratio",
}

missing_columns = required_columns - set(raw.columns)

if missing_columns:
    raise ValueError(
        f"Kolom raw tidak lengkap: {sorted(missing_columns)}"
    )


expected_rows = (
    raw["pair_id"].nunique()
    * raw["depart_at"].nunique()
)

actual_unique_rows = (
    raw[
        [
            "pair_id",
            "depart_at",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

coverage_pct = (
    100.0
    * actual_unique_rows
    / expected_rows
)

quality_audit = pd.DataFrame(
    {
        "metric": [
            "Expected pair-time observations",
            "Unique successful observations",
            "Coverage (%)",
            "Unique OD pairs",
            "Unique departure timestamps",
            "Missing historic_ratio",
            "Non-positive no_traffic_sec",
        ],
        "value": [
            expected_rows,
            actual_unique_rows,
            coverage_pct,
            raw["pair_id"].nunique(),
            raw["depart_at"].nunique(),
            raw["historic_ratio"].isna().sum(),
            (raw["no_traffic_sec"] <= 0).sum(),
        ],
    }
)

display(quality_audit)

if coverage_pct < 95:
    print(
        "PERINGATAN: coverage di bawah 95%. "
        "Periksa error sebelum menyimpulkan."
    )
else:
    print("Coverage memadai.")

## 7. Global pooled empirical quantiles

In [ ]:
valid = raw[
    raw["historic_ratio"].notna()
    & np.isfinite(raw["historic_ratio"])
    & (raw["historic_ratio"] > 0)
].copy()

q50_raw = valid["historic_ratio"].quantile(0.50)
q85_raw = valid["historic_ratio"].quantile(0.85)
q95_raw = valid["historic_ratio"].quantile(0.95)

empirical_multipliers = {
    "p50": 1.0,
    "p85": q85_raw / q50_raw,
    "p95": q95_raw / q50_raw,
}

quantile_summary = pd.DataFrame(
    {
        "quantile": [
            "p50",
            "p85",
            "p95",
        ],
        "raw_historic_ratio_quantile": [
            q50_raw,
            q85_raw,
            q95_raw,
        ],
        "normalized_empirical_multiplier": [
            empirical_multipliers["p50"],
            empirical_multipliers["p85"],
            empirical_multipliers["p95"],
        ],
    }
)

display(quantile_summary.round(6))

print("\nNORMALIZED EMPIRICAL MULTIPLIER")
print(
    f"p50 = "
    f"{empirical_multipliers['p50']:.4f}"
)
print(
    f"p85 = "
    f"{empirical_multipliers['p85']:.4f}"
)
print(
    f"p95 = "
    f"{empirical_multipliers['p95']:.4f}"
)

## 8. OD-pair cluster bootstrap 5.000 iterasi

In [ ]:
def normalized_quantile_multipliers(values):
    values = (
        pd.Series(values)
        .dropna()
        .to_numpy(dtype=float)
    )

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    p50 = np.quantile(values, 0.50)

    if p50 <= 0:
        return np.nan, np.nan, np.nan

    return (
        1.0,
        np.quantile(values, 0.85) / p50,
        np.quantile(values, 0.95) / p50,
    )


rng = np.random.default_rng(BOOTSTRAP_SEED)

pair_ids = (
    valid["pair_id"]
    .drop_duplicates()
    .to_numpy()
)

bootstrap_records = []


for iteration in tqdm(
    range(BOOTSTRAP_ITERATIONS),
    desc="OD-pair cluster bootstrap",
):
    sampled_pair_ids = rng.choice(
        pair_ids,
        size=len(pair_ids),
        replace=True,
    )

    sampled_clusters = [
        valid.loc[
            valid["pair_id"] == pair_id,
            "historic_ratio",
        ]
        for pair_id in sampled_pair_ids
    ]

    boot_values = pd.concat(
        sampled_clusters,
        ignore_index=True,
    )

    m50, m85, m95 = (
        normalized_quantile_multipliers(
            boot_values
        )
    )

    bootstrap_records.append(
        {
            "iteration": iteration + 1,
            "p50": m50,
            "p85": m85,
            "p95": m95,
        }
    )


bootstrap_df = pd.DataFrame(
    bootstrap_records
)

bootstrap_ci = pd.DataFrame(
    [
        {
            "quantile": q,
            "point_estimate": (
                empirical_multipliers[q]
            ),
            "bootstrap_mean": (
                bootstrap_df[q].mean()
            ),
            "ci95_lower": (
                bootstrap_df[q].quantile(0.025)
            ),
            "ci95_upper": (
                bootstrap_df[q].quantile(0.975)
            ),
        }
        for q in [
            "p50",
            "p85",
            "p95",
        ]
    ]
)

display(bootstrap_ci.round(6))

## 9. Ringkasan menurut jam dan tipe OD

In [ ]:
hourly_summary = (
    valid.groupby(
        "hour",
        as_index=False,
    )
    .agg(
        n=("historic_ratio", "size"),
        median_ratio=(
            "historic_ratio",
            "median",
        ),
        p85_ratio=(
            "historic_ratio",
            lambda x: x.quantile(0.85),
        ),
        p95_ratio=(
            "historic_ratio",
            lambda x: x.quantile(0.95),
        ),
    )
)

pair_type_summary = (
    valid.groupby(
        "pair_type",
        as_index=False,
    )
    .agg(
        n=("historic_ratio", "size"),
        median_ratio=(
            "historic_ratio",
            "median",
        ),
        p85_ratio=(
            "historic_ratio",
            lambda x: x.quantile(0.85),
        ),
        p95_ratio=(
            "historic_ratio",
            lambda x: x.quantile(0.95),
        ),
    )
)

display(hourly_summary.round(5))
display(pair_type_summary.round(5))

## 10. Visualisasi distribusi global

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    valid["historic_ratio"],
    bins=40,
)

plt.axvline(
    q50_raw,
    linestyle="--",
    label=f"Raw p50={q50_raw:.3f}",
)

plt.axvline(
    q85_raw,
    linestyle="--",
    label=f"Raw p85={q85_raw:.3f}",
)

plt.axvline(
    q95_raw,
    linestyle="--",
    label=f"Raw p95={q95_raw:.3f}",
)

plt.xlabel(
    "Historic Traffic / No-Traffic Ratio"
)

plt.ylabel("Frequency")

plt.title(
    "TomTom Global Pooled Historic Traffic Ratio Distribution"
)

plt.legend()
plt.tight_layout()

distribution_plot = (
    OUTPUT_DIR
    / "tomtom_historic_ratio_distribution.png"
)

plt.savefig(
    distribution_plot,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", distribution_plot)

## 11. Simpan output audit dan hasil empiris

In [ ]:
run_timestamp_utc = datetime.now(
    timezone.utc
).isoformat()

audit_metadata = pd.DataFrame(
    {
        "parameter": [
            "run_timestamp_utc",
            "input_mode",
            "source_file",
            "source_sha256",
            "api_product",
            "api_version",
            "computeTravelTimeFor",
            "primary_ratio",
            "aggregation",
            "normalization",
            "travel_mode",
            "route_type",
            "random_seed",
            "distance_strata",
            "customers_per_stratum",
            "representative_dates",
            "departure_hours",
            "od_pairs",
            "expected_observations",
            "successful_unique_observations",
            "coverage_pct",
            "bootstrap_type",
            "bootstrap_iterations",
            "bootstrap_seed",
        ],
        "value": [
            run_timestamp_utc,
            INPUT_MODE,
            os.path.basename(INPUT_FILE),
            SOURCE_SHA256,
            "TomTom Routing API",
            "v1",
            "all",
            (
                "historicTrafficTravelTimeInSeconds "
                "/ noTrafficTravelTimeInSeconds"
            ),
            "Global pooled empirical quantiles",
            "Qq(R) / Q0.50(R)",
            TRAVEL_MODE,
            ROUTE_TYPE,
            RANDOM_SEED,
            N_DISTANCE_STRATA,
            CUSTOMERS_PER_STRATUM,
            json.dumps(REPRESENTATIVE_DATES),
            json.dumps(DEPARTURE_HOURS),
            raw["pair_id"].nunique(),
            expected_rows,
            actual_unique_rows,
            coverage_pct,
            "OD-pair cluster bootstrap",
            BOOTSTRAP_ITERATIONS,
            BOOTSTRAP_SEED,
        ],
    }
)

raw.to_csv(
    OUTPUT_DIR
    / "tomtom_raw_validation.csv",
    index=False,
)

quantile_summary.to_csv(
    OUTPUT_DIR
    / "tomtom_empirical_quantiles.csv",
    index=False,
)

bootstrap_ci.to_csv(
    OUTPUT_DIR
    / "tomtom_bootstrap_ci95.csv",
    index=False,
)

hourly_summary.to_csv(
    OUTPUT_DIR
    / "tomtom_hourly_ratio_summary.csv",
    index=False,
)

pair_type_summary.to_csv(
    OUTPUT_DIR
    / "tomtom_pair_type_ratio_summary.csv",
    index=False,
)

audit_metadata.to_csv(
    OUTPUT_DIR
    / "tomtom_validation_audit_metadata.csv",
    index=False,
)

excel_output = (
    OUTPUT_DIR
    / "TomTom_Global_Pooled_Empirical_Quantiles.xlsx"
)

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl",
) as writer:

    quantile_summary.to_excel(
        writer,
        sheet_name="Empirical_Quantiles",
        index=False,
    )

    bootstrap_ci.to_excel(
        writer,
        sheet_name="Bootstrap_CI95",
        index=False,
    )

    quality_audit.to_excel(
        writer,
        sheet_name="Coverage_Audit",
        index=False,
    )

    hourly_summary.to_excel(
        writer,
        sheet_name="Hourly_Summary",
        index=False,
    )

    pair_type_summary.to_excel(
        writer,
        sheet_name="PairType_Summary",
        index=False,
    )

    od_pairs.to_excel(
        writer,
        sheet_name="Representative_OD",
        index=False,
    )

    raw.to_excel(
        writer,
        sheet_name="Raw_TomTom",
        index=False,
    )

    audit_metadata.to_excel(
        writer,
        sheet_name="Audit_Metadata",
        index=False,
    )

print("Excel output:", excel_output)

print("\nFiles:")
for file_path in sorted(
    OUTPUT_DIR.iterdir()
):
    if file_path.is_file():
        print("-", file_path.name)

## 12. Narasi hasil otomatis untuk Bab 3 dan Bab 4

In [ ]:
p85_ci_row = (
    bootstrap_ci
    .set_index("quantile")
    .loc["p85"]
)

p95_ci_row = (
    bootstrap_ci
    .set_index("quantile")
    .loc["p95"]
)

method_text = f"""
METODOLOGI TRAFFIC MULTIPLIER

Evaluasi variasi waktu perjalanan dilakukan menggunakan historical traffic
travel time dan no-traffic travel time TomTom. Sebanyak
{raw['pair_id'].nunique()} pasangan origin-destination representatif diamati pada
{raw['depart_at'].nunique()} kombinasi waktu keberangkatan sehingga diperoleh
{actual_unique_rows:,} observasi perjalanan. Rasio historical traffic travel time
terhadap no-traffic travel time dihitung pada setiap observasi. Seluruh rasio
kemudian digabungkan menjadi global pooled empirical distribution. Kuantil p50,
p85, dan p95 dihitung dari distribusi tersebut dan dinormalisasi terhadap median
empiris. Ketidakpastian estimasi dievaluasi menggunakan OD-pair cluster bootstrap
sebanyak {BOOTSTRAP_ITERATIONS:,} iterasi.
""".strip()

result_text = f"""
HASIL TRAFFIC MULTIPLIER

Hasil global pooled empirical quantiles menghasilkan normalized multiplier
sebesar {empirical_multipliers['p50']:.4f} pada p50,
{empirical_multipliers['p85']:.4f} pada p85, dan
{empirical_multipliers['p95']:.4f} pada p95. Interval empiris 95% hasil OD-pair
cluster bootstrap adalah [{p85_ci_row['ci95_lower']:.4f},
{p85_ci_row['ci95_upper']:.4f}] untuk p85 dan
[{p95_ci_row['ci95_lower']:.4f}, {p95_ci_row['ci95_upper']:.4f}] untuk p95.
""".strip()

print(method_text)
print("\n" + "=" * 80 + "\n")
print(result_text)

with open(
    OUTPUT_DIR
    / "narasi_tomtom_global_pooled.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(method_text)
    f.write("\n\n")
    f.write(result_text)

## 13. Download seluruh output

In [ ]:
zip_file = shutil.make_archive(
    "TomTom_Global_Pooled_Empirical_Quantiles_Output",
    "zip",
    root_dir=OUTPUT_DIR,
)

print("ZIP created:", zip_file)

try:
    from google.colab import files
    files.download(zip_file)

except ImportError:
    print(
        "File tersedia di:",
        Path(zip_file).resolve(),
    )

# Interpretasi akhir

Bila notebook dijalankan menggunakan raw output RKP yang sama, hasil normalized global pooled empirical multiplier adalah:

- p50 = **1.0000**
- p85 = **1.1559**
- p95 = **1.2935**

Nilai tersebut merupakan hasil langsung dari global pooled empirical quantiles rasio historical traffic travel time terhadap no-traffic travel time TomTom, setelah normalisasi terhadap median empiris.

Notebook ini merupakan artefak komputasional pendukung **Subbab 3.6.4**, **Subbab 4.3**, dan **Lampiran F**.